# MuonClip-RMS versus AdamW: 10-seed performance and WeightWatcher comparison

This lightweight post-training notebook compares the matched
MuonClip-RMS and AdamW arms of the 100-epoch, 10-seed experiment.
It reads only the persisted performance and WeightWatcher CSVs;
it does not load checkpoints, retrain models, or rerun
WeightWatcher.

Every trajectory shows the mean across independent seeded runs and
a Bollinger-style band equal to **mean plus or minus two sample
standard deviations across seeds**. These bands describe run-to-run
dispersion. They are not standard errors or confidence intervals.

The default WeightWatcher series is the training protocol's
primary `clip_xmax` fit (`fix_fingers=clip_xmax`). Failed fits are
excluded from the alpha mean and band, while their reduced seed
counts are retained in an availability table.

**`operator_kind`: `saved_multiseed_performance_and_default_weightwatcher_comparison`**

**`map_definition`: `identity read of completed-run CSV rows followed by cross-seed mean and sample-standard-deviation aggregation`**


In [ ]:
# Papermill parameters. Override these values in an injected cell.
RUN_ROOT = ""
OUTPUT_ROOT = ""
CHECKPOINT_CACHE_ROOT = ""
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
PROTOCOL_SLUG = ""
SEEDS = [1337, 2027, 31415]
CHECKPOINT_PAYLOAD_CACHE_SIZE = 24
SHOW_PLOTS = True
REQUIRE_ARTIFACTS = True
ALLOW_TEMPORARY_LONG_RUN = False
PROTOCOL_SLUG = "mnist_mlp3_tangent_rg_v1_muonclip_short100_10seed"
SEEDS = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010]
OPTIMIZER_SLUGS = ["muonclip_rms", "adamw"]
OPTIMIZER_LABELS = {
    "muonclip_rms": "MuonClip-RMS",
    "adamw": "AdamW",
}
EXPECTED_WEIGHT_LAYERS = ["fc1.weight", "fc2.weight", "fc3.weight"]
PRIMARY_FIT_VARIANT = "clip_xmax"
BAND_STD_MULTIPLIER = 2.0
ACCURACY_Y_MIN = 0.90
ACCURACY_Y_MAX = 1.005
METHOD_SLUG = "muonclip_adamw_10seed_bollinger_comparison"


In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from functools import lru_cache
import inspect
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "baseline" / "rg_baselines").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find baseline/rg_baselines. Launch Jupyter from a clone of "
        "CalculatedContent/rg_optimizers."
    )
BASELINE_ROOT = REPO_ROOT / "baseline"
EXPERIMENT_ROOT = BASELINE_ROOT / "experiments" / "mnist_mlp3_tangent_rg"
if str(BASELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASELINE_ROOT))

default_root = os.environ.get(
    "RG_MNIST_TANGENT_ROOT", "/tmp/rg-mnist-mlp3-tangent-rg"
)
RUN_ROOT_PATH = Path(RUN_ROOT or default_root).expanduser().resolve()

default_checkpoint_cache_root = os.environ.get(
    "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT",
    "/tmp/rg-mnist-mlp3-tangent-checkpoints",
)
CHECKPOINT_CACHE_ROOT_PATH = Path(
    CHECKPOINT_CACHE_ROOT or default_checkpoint_cache_root
).expanduser().resolve()

def _suite_name_from_profile():
    if str(PROTOCOL_SLUG).strip():
        return str(PROTOCOL_SLUG).strip()
    candidate = (
        Path(CONFIG_PATH).expanduser()
        if str(CONFIG_PATH).strip()
        else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
    )
    if candidate.is_file():
        if candidate.suffix.lower() == ".json":
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            value = payload.get("protocol", {}).get("suite_name")
            if value:
                return str(value)
        else:
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if stripped.startswith("suite_name:"):
                    return stripped.split(":", 1)[1].strip().strip("'\"")
    fallback = {
        "smoke": "mnist_mlp3_tangent_rg_v1_smoke",
        "pilot_1000_epochs": "mnist_mlp3_tangent_rg_v1_pilot1000",
        "long_horizon_10000_epochs": "mnist_mlp3_tangent_rg_v1_reference10000",
    }
    if PROFILE not in fallback:
        raise FileNotFoundError(
            f"Cannot derive suite_name for PROFILE={PROFILE!r}; set CONFIG_PATH "
            "or PROTOCOL_SLUG explicitly."
        )
    return fallback[PROFILE]

PROTOCOL_SLUG = _suite_name_from_profile()
OUTPUT_ROOT_PATH = Path(
    OUTPUT_ROOT or RUN_ROOT_PATH / PROTOCOL_SLUG / "notebook_outputs"
).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)

SEEDS = tuple(int(seed) for seed in SEEDS)
if SEEDS != (1337, 2027, 31415):
    print("WARNING: this is not the preregistered three-seed tuple:", SEEDS)

print("repository:", REPO_ROOT)
print("run root:", RUN_ROOT_PATH)
print("tail checkpoint cache root:", CHECKPOINT_CACHE_ROOT_PATH)
print("effective suite:", PROTOCOL_SLUG)
print("output root:", OUTPUT_ROOT_PATH)
print("seeds:", SEEDS)


In [ ]:
from matplotlib.lines import Line2D


In [ ]:
def require_path(path, *, description="artifact"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path


def boolean_series(values):
    if getattr(values, "dtype", None) == bool:
        return values
    return values.astype(str).str.strip().str.lower().isin(
        {"1", "true", "yes"}
    )


def resolve_seed_dir(optimizer_slug, seed):
    protocol = RUN_ROOT_PATH / PROTOCOL_SLUG
    if not protocol.is_dir():
        protocol = RUN_ROOT_PATH
    candidates = [
        protocol / optimizer_slug / f"seed_{int(seed)}",
        protocol / optimizer_slug / f"seed_{int(seed):05d}",
        protocol / "results" / optimizer_slug / f"seed_{int(seed)}",
        protocol / "results" / optimizer_slug / f"seed_{int(seed):05d}",
        RUN_ROOT_PATH / optimizer_slug / f"seed_{int(seed)}",
        RUN_ROOT_PATH / "results" / optimizer_slug / f"seed_{int(seed)}",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Missing completed seed directory for optimizer={optimizer_slug}, "
        f"seed={seed}. Checked:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


def validate_run_identity(seed_dir, *, optimizer_slug, seed):
    seed_dir = Path(seed_dir)
    manifest = json.loads(
        require_path(seed_dir / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    resolved = json.loads(
        require_path(
            seed_dir / "resolved_config.json",
            description="resolved configuration",
        ).read_text(encoding="utf-8")
    )
    completion = json.loads(
        require_path(
            seed_dir / "run_complete.json",
            description="completion marker",
        ).read_text(encoding="utf-8")
    )
    config = dict(resolved.get("config", resolved))
    expected = {
        "suite": str(PROTOCOL_SLUG),
        "optimizer": str(optimizer_slug),
        "seed": int(seed),
    }
    observed = {
        "suite": str(manifest.get("suite_name")),
        "optimizer": str(manifest.get("optimizer")),
        "seed": int(manifest.get("seed", -1)),
    }
    failures = []
    if observed != expected:
        failures.append(f"manifest identity={observed}, expected={expected}")
    if str(config.get("suite_name")) != expected["suite"]:
        failures.append("resolved suite does not match")
    if str(config.get("optimizer")) != expected["optimizer"]:
        failures.append("resolved optimizer does not match")
    if int(config.get("seed", -1)) != expected["seed"]:
        failures.append("resolved seed does not match")
    if str(completion.get("optimizer")) != expected["optimizer"]:
        failures.append("completion optimizer does not match")
    if int(completion.get("seed", -1)) != expected["seed"]:
        failures.append("completion seed does not match")
    if completion.get("completed") is not True:
        failures.append("completion marker does not declare completed=true")
    fingerprints = {
        str(manifest.get("protocol_fingerprint", "")),
        str(resolved.get("protocol_fingerprint", "")),
        str(completion.get("protocol_fingerprint", "")),
    }
    if "" in fingerprints or len(fingerprints) != 1:
        failures.append("manifest/resolved/completion fingerprints disagree")
    if int(completion.get("epochs", -1)) != int(config.get("epochs", -2)):
        failures.append("completion and resolved epoch horizons disagree")
    if failures:
        raise RuntimeError(
            f"Run identity failure beneath {seed_dir}:\n  - "
            + "\n  - ".join(failures)
        )
    return manifest, resolved, completion, fingerprints.pop()


def bollinger_summary(frame, *, groups, value, multiplier):
    summary = (
        frame.groupby(list(groups), as_index=False, dropna=False)[value]
        .agg(n="count", mean="mean", std="std")
    )
    summary["band_multiplier"] = float(multiplier)
    summary["band_low"] = summary["mean"] - float(multiplier) * summary["std"]
    summary["band_high"] = summary["mean"] + float(multiplier) * summary["std"]
    return summary


## Load and verify the matched completed runs

Both optimizer arms must contain every requested seed and the same
epoch grid. Dataset split hashes, architecture, initialization,
analysis schedule, device, and software provenance must also agree.


In [ ]:
OPTIMIZER_SLUGS = tuple(str(value) for value in OPTIMIZER_SLUGS)
EXPECTED_WEIGHT_LAYERS = tuple(str(value) for value in EXPECTED_WEIGHT_LAYERS)
ACTIVE_SEEDS = tuple(int(value) for value in SEEDS)
if len(ACTIVE_SEEDS) != 10 or len(set(ACTIVE_SEEDS)) != 10:
    raise ValueError(f"Expected exactly 10 unique seeds; observed {ACTIVE_SEEDS}")
if set(OPTIMIZER_SLUGS) != {"muonclip_rms", "adamw"}:
    raise ValueError(
        "OPTIMIZER_SLUGS must contain matched muonclip_rms and adamw arms"
    )
if str(PRIMARY_FIT_VARIANT) != "clip_xmax":
    raise ValueError(
        "The default WeightWatcher comparison is preregistered as clip_xmax"
    )
if not np.isclose(float(BAND_STD_MULTIPLIER), 2.0):
    raise ValueError("Bollinger bands must remain mean +/- 2 sample SD")
if not (
    0.0 <= float(ACCURACY_Y_MIN)
    < float(ACCURACY_Y_MAX)
    <= 1.01
):
    raise ValueError(
        "Accuracy y-axis limits must satisfy "
        "0 <= ACCURACY_Y_MIN < ACCURACY_Y_MAX <= 1.01"
    )

required_performance = {
    "optimizer", "seed", "epoch", "global_step", "protocol_fingerprint",
    "train_loss", "train_accuracy", "test_loss", "test_accuracy",
    "test_monitoring_only", "test_used_for_selection",
}
required_weightwatcher = {
    "optimizer", "seed", "epoch", "global_step", "protocol_fingerprint",
    "layer", "fit_variant", "finger_policy", "alpha", "fit_ok",
    "operator_kind", "map_definition",
}

performance_frames = []
weightwatcher_frames = []
provenance_rows = []
invariant_snapshots = []
for optimizer_slug in OPTIMIZER_SLUGS:
    for seed in ACTIVE_SEEDS:
        seed_dir = resolve_seed_dir(optimizer_slug, seed)
        manifest, resolved, completion, fingerprint = validate_run_identity(
            seed_dir, optimizer_slug=optimizer_slug, seed=seed
        )
        metrics_dir = require_path(seed_dir / "metrics", description="metrics directory")
        performance_path = require_path(
            metrics_dir / "performance_by_analysis_epoch.csv",
            description="saved performance table",
        )
        weightwatcher_path = require_path(
            metrics_dir / "weightwatcher_fits.csv",
            description="saved WeightWatcher table",
        )
        performance_run = pd.read_csv(performance_path)
        weightwatcher_run = pd.read_csv(weightwatcher_path)
        for label, frame, required in (
            ("performance", performance_run, required_performance),
            ("WeightWatcher", weightwatcher_run, required_weightwatcher),
        ):
            missing = required - set(frame.columns)
            if missing:
                raise RuntimeError(
                    f"{label} table {seed_dir} lacks columns {sorted(missing)}"
                )
            if frame.empty:
                raise RuntimeError(f"{label} table is empty beneath {seed_dir}")
            if set(frame["optimizer"].astype(str)) != {optimizer_slug}:
                raise RuntimeError(f"{label} optimizer mismatch beneath {seed_dir}")
            if set(pd.to_numeric(frame["seed"]).astype(int)) != {seed}:
                raise RuntimeError(f"{label} seed mismatch beneath {seed_dir}")
            if set(frame["protocol_fingerprint"].astype(str)) != {fingerprint}:
                raise RuntimeError(f"{label} fingerprint mismatch beneath {seed_dir}")
        performance_frames.append(performance_run)
        weightwatcher_frames.append(weightwatcher_run)
        provenance_rows.append({
            "optimizer": optimizer_slug,
            "seed": seed,
            "protocol_fingerprint": fingerprint,
            "seed_dir": str(seed_dir.resolve()),
            "performance_path": str(performance_path.resolve()),
            "weightwatcher_path": str(weightwatcher_path.resolve()),
            "completion_epoch": int(completion["epochs"]),
        })
        invariant_snapshots.append({
            "optimizer": optimizer_slug,
            "seed": seed,
            "suite_name": manifest.get("suite_name"),
            "dataset": manifest.get("dataset"),
            "model": manifest.get("model"),
            "initialization": manifest.get("initialization"),
            "normalization": manifest.get("normalization"),
            "train_indices_sha256": manifest.get("train_indices_sha256"),
            "validation_indices_sha256": manifest.get("validation_indices_sha256"),
            "analysis_plan": manifest.get("analysis_plan"),
            "device": manifest.get("device"),
            "software_versions": manifest.get("software_versions"),
            "determinism_settings": manifest.get("determinism_settings"),
            "test_monitoring_only": manifest.get("test_monitoring_only"),
        })

invariant_fields = [
    "suite_name", "dataset", "model", "initialization", "normalization",
    "train_indices_sha256", "validation_indices_sha256", "analysis_plan",
    "device", "software_versions", "determinism_settings",
    "test_monitoring_only",
]
disagreements = []
for field in invariant_fields:
    values = {
        json.dumps(row[field], sort_keys=True, default=str)
        for row in invariant_snapshots
    }
    if len(values) != 1:
        disagreements.append(field)
if disagreements:
    raise RuntimeError(
        "Matched optimizer/seed runs disagree on provenance: "
        + ", ".join(disagreements)
    )

performance = pd.concat(performance_frames, ignore_index=True, sort=False)
weightwatcher = pd.concat(weightwatcher_frames, ignore_index=True, sort=False)
provenance = pd.DataFrame(provenance_rows)
for column in (
    "epoch", "global_step", "train_loss", "train_accuracy",
    "test_loss", "test_accuracy",
):
    performance[column] = pd.to_numeric(performance[column], errors="coerce")
for column in ("epoch", "global_step", "alpha"):
    weightwatcher[column] = pd.to_numeric(weightwatcher[column], errors="coerce")
weightwatcher["fit_ok_bool"] = boolean_series(weightwatcher["fit_ok"])

expected_optimizer_seed_grid = {
    (optimizer, seed)
    for optimizer in OPTIMIZER_SLUGS
    for seed in ACTIVE_SEEDS
}
observed_performance_grid = set(
    performance[["optimizer", "seed"]]
    .drop_duplicates()
    .assign(seed=lambda frame: frame["seed"].astype(int))
    .itertuples(index=False, name=None)
)
observed_weightwatcher_grid = set(
    weightwatcher[["optimizer", "seed"]]
    .drop_duplicates()
    .assign(seed=lambda frame: frame["seed"].astype(int))
    .itertuples(index=False, name=None)
)
if observed_performance_grid != expected_optimizer_seed_grid:
    raise RuntimeError("Performance optimizer/seed grid is incomplete")
if observed_weightwatcher_grid != expected_optimizer_seed_grid:
    raise RuntimeError("WeightWatcher optimizer/seed grid is incomplete")
if performance.duplicated(["optimizer", "seed", "epoch", "global_step"]).any():
    raise RuntimeError("Duplicate performance state rows were found")
if weightwatcher.duplicated(
    ["optimizer", "seed", "epoch", "global_step", "layer", "fit_variant"]
).any():
    raise RuntimeError("Duplicate WeightWatcher state/layer/variant rows were found")

epoch_grids = {
    (optimizer, seed): tuple(
        run.sort_values("epoch")["epoch"].astype(int).tolist()
    )
    for (optimizer, seed), run in performance.groupby(["optimizer", "seed"])
}
if len(set(epoch_grids.values())) != 1:
    raise RuntimeError("Completed runs do not share an identical performance epoch grid")

comparison_root = OUTPUT_ROOT_PATH / METHOD_SLUG
figure_root = comparison_root / "figures"
figure_root.mkdir(parents=True, exist_ok=True)
performance.to_csv(comparison_root / "performance_rows_used.csv", index=False)
weightwatcher.to_csv(comparison_root / "weightwatcher_rows_used.csv", index=False)
provenance.to_csv(comparison_root / "cross_run_provenance.csv", index=False)
print("performance rows:", len(performance))
print("WeightWatcher rows:", len(weightwatcher))
display(provenance)


## Aggregate independent seeds

The standard deviation is computed across the ten complete runs at
each epoch. It is not divided by the square root of the seed count.
For accuracy only, plotted band endpoints are clipped to the
physical interval $[0,1]$; the unmodified numeric bands remain in
the saved CSV.


In [ ]:
performance_long = performance.melt(
    id_vars=["optimizer", "seed", "epoch", "global_step"],
    value_vars=["train_loss", "test_loss", "train_accuracy", "test_accuracy"],
    var_name="metric",
    value_name="value",
)
if not np.isfinite(performance_long["value"].to_numpy(dtype=float)).all():
    raise RuntimeError("Performance comparison contains non-finite values")
performance_summary = bollinger_summary(
    performance_long,
    groups=("optimizer", "epoch", "metric"),
    value="value",
    multiplier=BAND_STD_MULTIPLIER,
)
if not performance_summary["n"].eq(len(ACTIVE_SEEDS)).all():
    raise RuntimeError("A performance Bollinger group is missing one or more seeds")

default_weightwatcher = weightwatcher[
    weightwatcher["fit_variant"].astype(str).eq(PRIMARY_FIT_VARIANT)
    & weightwatcher["layer"].astype(str).isin(EXPECTED_WEIGHT_LAYERS)
].copy()
expected_finger_policy = set(
    default_weightwatcher["finger_policy"].dropna().astype(str)
)
if expected_finger_policy != {"fix_fingers=clip_xmax"}:
    raise RuntimeError(
        f"Default WeightWatcher finger policy mismatch: {expected_finger_policy}"
    )
default_weightwatcher["alpha_for_summary"] = default_weightwatcher["alpha"].where(
    default_weightwatcher["fit_ok_bool"]
    & np.isfinite(default_weightwatcher["alpha"])
    & default_weightwatcher["alpha"].gt(0.0)
)
alpha_summary = bollinger_summary(
    default_weightwatcher,
    groups=("optimizer", "epoch", "layer"),
    value="alpha_for_summary",
    multiplier=BAND_STD_MULTIPLIER,
)
alpha_availability = (
    default_weightwatcher.groupby(
        ["optimizer", "epoch", "global_step", "layer"], as_index=False
    )
    .agg(
        requested_seed_count=("seed", "nunique"),
        successful_seed_count=("alpha_for_summary", "count"),
    )
)
if not alpha_availability["requested_seed_count"].eq(len(ACTIVE_SEEDS)).all():
    raise RuntimeError("Default WeightWatcher grid is missing requested seed rows")

performance_summary.to_csv(
    comparison_root / "performance_bollinger_summary.csv", index=False
)
alpha_summary.to_csv(
    comparison_root / "default_weightwatcher_alpha_bollinger_summary.csv",
    index=False,
)
alpha_availability.to_csv(
    comparison_root / "default_weightwatcher_fit_availability.csv", index=False
)
display(performance_summary.head(12))
display(alpha_availability.groupby(["optimizer", "layer"], as_index=False).agg(
    minimum_successful_seeds=("successful_seed_count", "min"),
    maximum_successful_seeds=("successful_seed_count", "max"),
))


## Training and test accuracy and loss

Thin lines are individual runs. Thick lines are cross-seed means;
shading is the Bollinger-style mean plus or minus two seed standard
deviations.


In [ ]:
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

optimizer_style = {
    "muonclip_rms": {"label": "MuonClip-RMS", "color": "#0072B2"},
    "adamw": {"label": "AdamW", "color": "#D55E00"},
}

def save_figure(fig, filename, *, bottom=0.0):
    fig.tight_layout(rect=(0, bottom, 1, 1))
    target = figure_root / filename
    fig.savefig(target, dpi=190, bbox_inches="tight", facecolor="white")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return target


def plot_performance_pair(metrics, *, ylabel, title, filename, bounded=False):
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.7), sharex=True)
    for axis, metric in zip(axes, metrics):
        raw_metric = performance_long[
            performance_long["metric"].eq(metric)
            & performance_long["epoch"].gt(0)
        ]
        summary_metric = performance_summary[
            performance_summary["metric"].eq(metric)
            & performance_summary["epoch"].gt(0)
        ]
        for optimizer in OPTIMIZER_SLUGS:
            style = optimizer_style[optimizer]
            raw_optimizer = raw_metric[raw_metric["optimizer"].eq(optimizer)]
            for _, seed_curve in raw_optimizer.groupby("seed"):
                seed_curve = seed_curve.sort_values("epoch")
                axis.plot(
                    seed_curve["epoch"], seed_curve["value"],
                    color=style["color"], alpha=0.10, linewidth=0.65,
                )
            curve = summary_metric[
                summary_metric["optimizer"].eq(optimizer)
            ].sort_values("epoch")
            x = curve["epoch"].to_numpy(dtype=float)
            mean = curve["mean"].to_numpy(dtype=float)
            low = curve["band_low"].to_numpy(dtype=float)
            high = curve["band_high"].to_numpy(dtype=float)
            if bounded:
                low = np.clip(low, 0.0, 1.0)
                high = np.clip(high, 0.0, 1.0)
            else:
                low = np.maximum(low, 0.0)
            axis.fill_between(
                x, low, high, color=style["color"], alpha=0.18, linewidth=0,
            )
            axis.plot(
                x, mean, color=style["color"], linewidth=2.2,
                label=style["label"],
            )
        split = metric.split("_", 1)[0].title()
        axis.set(
            xlabel="epoch",
            ylabel=ylabel,
            title=f"{split} {title}",
            xlim=(1, None),
        )
        if bounded:
            axis.set_ylim(
                float(ACCURACY_Y_MIN),
                float(ACCURACY_Y_MAX),
            )
        axis.grid(True, alpha=0.25)
        axis.legend(frameon=False)
    return save_figure(fig, filename)


accuracy_figure = plot_performance_pair(
    ("train_accuracy", "test_accuracy"),
    ylabel="accuracy",
    title="accuracy",
    filename="train_test_accuracy_bollinger_2sd.png",
    bounded=True,
)
loss_figure = plot_performance_pair(
    ("train_loss", "test_loss"),
    ylabel="cross-entropy loss",
    title="loss",
    filename="train_test_loss_bollinger_2sd.png",
    bounded=False,
)
print(accuracy_figure)
print(loss_figure)


## Default WeightWatcher alpha by layer

These panels use only successful persisted `clip_xmax` fits. The
availability CSV gives the exact contributing seed count at every
epoch and layer. The horizontal $alpha=2$ line is a reference,
not a parameter-selection objective.


In [ ]:
fig, axes = plt.subplots(
    1, len(EXPECTED_WEIGHT_LAYERS),
    figsize=(4.8 * len(EXPECTED_WEIGHT_LAYERS), 4.8),
    sharex=True,
)
axes = np.atleast_1d(axes)
for axis, layer in zip(axes, EXPECTED_WEIGHT_LAYERS):
    for optimizer in OPTIMIZER_SLUGS:
        style = optimizer_style[optimizer]
        curve = alpha_summary[
            alpha_summary["optimizer"].eq(optimizer)
            & alpha_summary["layer"].astype(str).eq(layer)
            & alpha_summary["n"].ge(2)
        ].sort_values("epoch")
        if curve.empty:
            continue
        x = curve["epoch"].to_numpy(dtype=float)
        mean = curve["mean"].to_numpy(dtype=float)
        low = curve["band_low"].to_numpy(dtype=float)
        high = curve["band_high"].to_numpy(dtype=float)
        axis.fill_between(
            x, low, high, color=style["color"], alpha=0.18, linewidth=0,
        )
        axis.plot(
            x, mean, color=style["color"], linewidth=2.1,
            label=style["label"],
        )
    axis.axhline(2.0, color="#333333", linestyle=":", linewidth=1.1)
    axis.set(
        xlabel="epoch",
        ylabel=r"WeightWatcher $\alpha$",
        title=layer.replace(".weight", ""),
    )
    axis.grid(True, alpha=0.25)
    axis.legend(frameon=False)
alpha_figure = save_figure(
    fig, "default_weightwatcher_alpha_by_layer_bollinger_2sd.png"
)
print(alpha_figure)


## Peak-to-final test degradation

The table is descriptive only: test metrics remain monitoring-only
and do not select checkpoints or hyperparameters.


In [ ]:
degradation_rows = []
for (optimizer, seed), run in performance.groupby(["optimizer", "seed"]):
    run = run.sort_values("epoch")
    peak_index = run["test_accuracy"].idxmax()
    minimum_loss_index = run["test_loss"].idxmin()
    peak = run.loc[peak_index]
    minimum_loss = run.loc[minimum_loss_index]
    final = run.iloc[-1]
    degradation_rows.append({
        "optimizer": optimizer,
        "seed": int(seed),
        "peak_test_accuracy": float(peak["test_accuracy"]),
        "peak_test_accuracy_epoch": int(peak["epoch"]),
        "final_test_accuracy": float(final["test_accuracy"]),
        "peak_to_final_test_accuracy_change": float(
            final["test_accuracy"] - peak["test_accuracy"]
        ),
        "minimum_test_loss": float(minimum_loss["test_loss"]),
        "minimum_test_loss_epoch": int(minimum_loss["epoch"]),
        "final_test_loss": float(final["test_loss"]),
        "minimum_to_final_test_loss_change": float(
            final["test_loss"] - minimum_loss["test_loss"]
        ),
    })
degradation = pd.DataFrame(degradation_rows)
degradation_summary = bollinger_summary(
    degradation.melt(
        id_vars=["optimizer", "seed"],
        value_vars=[
            "peak_to_final_test_accuracy_change",
            "minimum_to_final_test_loss_change",
        ],
        var_name="metric",
        value_name="value",
    ),
    groups=("optimizer", "metric"),
    value="value",
    multiplier=BAND_STD_MULTIPLIER,
)
degradation.to_csv(
    comparison_root / "per_seed_peak_to_final_degradation.csv", index=False
)
degradation_summary.to_csv(
    comparison_root / "peak_to_final_degradation_bollinger_summary.csv",
    index=False,
)
display(degradation_summary)


## Completion and provenance


In [ ]:
output_files = sorted(
    str(path.relative_to(comparison_root))
    for path in comparison_root.rglob("*")
    if path.is_file()
)
summary = {
    "schema_version": 1,
    "completed": True,
    "status": "complete",
    "suite_name": str(PROTOCOL_SLUG),
    "method_slug": str(METHOD_SLUG),
    "operator_kind": (
        "saved_multiseed_performance_and_default_weightwatcher_comparison"
    ),
    "map_definition": (
        "identity read of completed-run CSV rows followed by cross-seed "
        "mean and sample-standard-deviation aggregation"
    ),
    "optimizer_slugs": list(OPTIMIZER_SLUGS),
    "seeds": list(ACTIVE_SEEDS),
    "seed_count": len(ACTIVE_SEEDS),
    "primary_weightwatcher_fit_variant": str(PRIMARY_FIT_VARIANT),
    "finger_policy": "fix_fingers=clip_xmax",
    "uncertainty_policy": "bollinger_mean_plus_or_minus_2_sample_sd_across_seeds",
    "band_standard_deviation_multiplier": float(BAND_STD_MULTIPLIER),
    "performance_row_count": int(len(performance)),
    "weightwatcher_row_count": int(len(weightwatcher)),
    "output_files": output_files,
    "test_monitoring_only": True,
}
(comparison_root / "comparison_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(pd.DataFrame([summary]))
print("complete comparison:", comparison_root)
